# Anima 生图 + Wan 视频 - Colab 一次成型版

**这个版本把之前所有踩坑固化成自动逻辑，正常情况跑 1→2→3→4 四格就出图，不用再改任何东西。**

| 已固化的坑 | 处理方式 |
|---|---|
| Anima 在 T4 只能 fp32（bf16 训练的模型无 fp16 路） | GPU 档默认换 **animagine-xl-4.0 fp16**（实测 1024@28=32s），Anima 降为可选 `GPU_MODEL='anima'` |
| `load_components(torch_dtype=)` 被新版 diffusers 忽略 | 只给 `from_pretrained(dtype=)` 传（新版关键字），load_components 不传 |
| 832x1216 在 fp32 下 OOM | 服务端 OOM 自动降档重试（x0.8 缩，最多 3 次） |
| 事后 .half()/搬 CPU -> 黑图/device 错乱 | 不存在了：CPU 上一次性定精度再上 GPU |
| 90 分钟无操作断线 | cell 2 自动保活 |
| /content 会话结束清空 | cell 2 挂 Drive，cell 4 有 /flush |
| RAM 爆（二次加载 / CPU 凑 fp32） | cell 3 全程一次 fp16 加载（CPU 峰值 ~7GB），精度只在 GPU 原地转 |

**实测速度（T4）**：animagine 832x1216@28 ≈ 30s / 1024x1024@28 ≈ 32s；anima(fp32) 512x768@16 ≈ 64s。
**无审查**：裸权重，想画什么写什么。

In [ ]:
#@title 1. 环境探测 + 依赖（先跑这格）
import os, sys, subprocess, shutil
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')  # 抗显存碎片

IN_COLAB = os.path.isdir('/content')
print('colab:', IN_COLAB)

HAS_GPU = False; GPU_NAME = 'none'; VRAM_GB = 0.0
try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=30)
    if r.returncode == 0 and r.stdout.strip():
        GPU_NAME = r.stdout.split(',')[0].strip()
        VRAM_GB = float(r.stdout.split(',')[1].replace('MiB','').strip()) / 1024
        HAS_GPU = True
except Exception as e:
    print('nvidia-smi:', e)
print('GPU : %s %.1fGB' % (GPU_NAME, VRAM_GB) if HAS_GPU else 'GPU : 无（CPU 档 -> OpenVINO 生图）')

try:
    import psutil
    vm = psutil.virtual_memory()
    print('RAM : %.1f 总 / %.1f 空闲' % (vm.total/1024**3, vm.available/1024**3))
except ImportError:
    pass
print('disk: %.1fGB free' % (shutil.disk_usage('/content' if IN_COLAB else '.').free/1024**3))

if IN_COLAB:
    %pip -q install diffusers transformers accelerate safetensors flask flask-cors psutil pillow requests numpy
    # CPU 档现在走纯 diffusers fp16，不再需要 optimum-intel/openvino（旧路已被证明装不上）。
    # 想再试 OpenVINO 加速：把下面注释打开（注意 optimum.intel 的无门 import 会拉 open_clip/kokoro/timm）
    # if not HAS_GPU:
    #     %pip -q install optimum-intel openvino open_clip_torch timm sentence_transformers kokoro

import torch
if HAS_GPU:
    torch.backends.cuda.matmul.allow_tf32 = True   # T4 fp32 免费提速 ~2x
    torch.backends.cudnn.allow_tf32 = True
print('[+] 就绪 | torch', torch.__version__)

In [ ]:
#@title 2. 会话保活 + Drive 落盘
import os, time, shutil, threading

OUT_DIR = '/content/out' if IN_COLAB else './out'
os.makedirs(OUT_DIR, exist_ok=True)

if IN_COLAB:
    from IPython.display import display, Javascript
    display(Javascript("""
    function ka(){var b=document.querySelector('colab-connect-button');
      if(b&&b.shadowRoot){var c=b.shadowRoot.querySelector('#connect');if(c)c.click();}}
    setInterval(ka,60000);"""))
    print('[+] keepalive 60s')

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ai_out'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_OK = True
    print('[+] Drive ->', DRIVE_DIR)
except Exception as e:
    print('[!] 无 Drive（用浏览器下载也行）:', str(e)[:120])

def flush_to_drive(verbose=True):
    if not DRIVE_OK:
        if verbose: print('[!] Drive 不可用，产出在', OUT_DIR)
        return
    tgt = os.path.join(DRIVE_DIR, time.strftime('%m%d_%H%M'))
    os.makedirs(tgt, exist_ok=True)
    n = 0
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            shutil.copy2(os.path.join(root, f), os.path.join(tgt, f)); n += 1
    if verbose: print('[+] 搬了 %d 个文件 -> %s' % (n, tgt))

In [ ]:
#@title 3. 加载 GPU 模型（默认 animagine-xl-4.0 fp16，实测 1024@28步 ~32s）
# Anima 的硬件墙：bf16 训练的模型，T4 无 bf16 -> fp16 溢出黑图，只能 fp32(慢)。
# 所以 GPU 档默认换 SDXL 动漫底（fp16 原生安全）。改 GPU_MODEL 切换：
#   'animagine'  SDXL 动漫天花板，fp16 28步 ~30s/张          （推荐，已实测）
#   'nova_lcm'   SDXL 动漫 LCM 蒸馏，8步 ~10s/张（画质略降）
#   'anima'      Anima fp32，512x768@16 ~64s（那种质感要它，慢）
GPU_MODEL = 'animagine'
ANIMA_ID = 'circlestone-labs/Anima-Base-v1.0-Diffusers'
SDXL_IDS = {'animagine': 'cagliostrolab/animagine-xl-4.0', 'nova_lcm': 'sca255/nova_xl_lcm'}
GPU_DEFAULTS = {'animagine': {'w': 832, 'h': 1216, 'steps': 28, 'guidance': 7.0},
                'nova_lcm':  {'w': 1024, 'h': 1024, 'steps': 8,  'guidance': 1.5},
                'anima':     {'w': 512,  'h': 768,  'steps': 16, 'guidance': None}}
pipe = None

if not HAS_GPU:
    print('[!] CPU 档：本格跳过（cell 4 自动走 OpenVINO 生图）')
else:
    import torch, gc, numpy as np

    def _fp16(cls, mid, **kw):
        try:
            return cls.from_pretrained(mid, dtype=torch.float16, **kw)
        except TypeError:
            return cls.from_pretrained(mid, torch_dtype=torch.float16, **kw)

    def _load(name):
        global pipe
        if pipe is not None:
            del pipe
        gc.collect(); torch.cuda.empty_cache()
        if name == 'anima':
            from diffusers import ModularPipeline
            p = _fp16(ModularPipeline, ANIMA_ID)
            p.load_components()
            p.to('cuda')
            for a in ('text_encoder', 'text_conditioner', 'transformer', 'vae'):
                c = getattr(p, a, None)
                if c is not None:
                    try:
                        c.to(torch.float32)   # GPU 上原地升 fp32，CPU 里永远只有 fp16 一份
                    except Exception:
                        pass
        else:
            from diffusers import StableDiffusionXLPipeline
            p = _fp16(StableDiffusionXLPipeline, SDXL_IDS[name], use_safetensors=True)
            p.to('cuda')
        p.set_progress_bar_config(disable=True)
        try:
            p.enable_xformers_memory_efficient_attention()
        except Exception:
            try:
                p.enable_attention_slicing()
            except Exception:
                pass
        gc.collect(); torch.cuda.empty_cache()
        return p

    def _test(name):
        cfg = GPU_DEFAULTS[name]
        kw = dict(height=512, width=512, num_inference_steps=8,
                  generator=torch.Generator('cuda').manual_seed(7))
        if cfg['guidance'] is not None:
            kw['guidance_scale'] = cfg['guidance']
        out = pipe(prompt='masterpiece, best quality, 1girl, portrait',
                   negative_prompt='worst quality', **kw)
        m = float(np.asarray(out.images[0].convert('RGB')).mean())
        del out
        gc.collect(); torch.cuda.empty_cache()
        print('  [%s] 测试图 mean=%.1f -> %s' % (name, m, '真图' if m >= 6 else '黑图'))
        return m >= 6

    try:
        pipe = _load(GPU_MODEL)
        if _test(GPU_MODEL):
            free_b, total_b = torch.cuda.mem_get_info()
            c = GPU_DEFAULTS[GPU_MODEL]
            print('[+] GPU_MODEL=%s 就绪 | 显存空闲 %.1f/%.1f GB | 默认 %dx%d@%d步'
                  % (GPU_MODEL, free_b/1024**3, total_b/1024**3, c['w'], c['h'], c['steps']))
        else:
            print('[x] %s 黑图/异常 -> 改 GPU_MODEL 为其它档重跑本格' % GPU_MODEL)
            pipe = None
    except Exception as e:
        print('[x] cell3 异常: %s: %s' % (type(e).__name__, str(e)[:300]))

In [ ]:
#@title 4. 启动服务 + 隧道（DSH / CLI 直调入口）
import threading, uuid, os, io, json, time, re, queue, subprocess, urllib.request
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
import numpy as np

app = Flask(__name__); CORS(app)
JOBS = {}; LOCK = threading.Lock(); TASK_Q = queue.Queue()
URL = None

# ---- CPU 档预设（无 GPU 才用）----
# 2026-10-18 重写：彻底不碰 optimum-intel / OpenVINO。
# 根因（静态分析 optimum-intel 2.2.0 wheel 坐实）：
#   optimum/intel/openvino/__init__.py  L84 `from .modeling_open_clip import ...`
#                                       L97 `from .modeling_text2speech import ...`
#   这两行**没有 is_xxx_available() 门** → import optimum.intel 就硬拉
#   open_clip / kokoro / timm / sentence_transformers，Colab 镜像一个都没装
#   → 底层 ModuleNotFoundError 被 transformers._LazyModule 重写成
#     "Could not import module 'OVDiffusionPipeline'. Are this object's requirements defined correctly?"
#   与 diffusers / transformers / openvino 版本无关，换任何 cpu_preset 都炸 → 旧 CPU 档整条路死。
# 新方案：纯 diffusers fp16 + enable_model_cpu_offload（RAM 峰值只多一个组件）。
#   engine 可选：'diffusers'(默认稳) | 'openvino'(显式尝试，失败自动降级)
CPU_ENGINE = 'diffusers'
CPU_SLICING = False   # True = 开 attention/vae slicing（更省内存但明显更慢）
# ram_gb: pipeline 权重实测体积（HF blobs API，只算子目录 safetensors，不含根目录 single-file ckpt）。
#   只有 fp16 权重的仓库（sdxl_anime/nova_xl_lcm）要 fp32 就得下载后转精度 → 峰值翻倍，故 float32 预算按 2x 填。
#   超闸门就不加载，避免把 Colab 会话打死（12.6GB RAM 墙）。
CPU_PRESETS = {
    'sd_turbo':    {'cls':'StableDiffusionPipeline','model':'stabilityai/sd-turbo','steps':4,'guidance':0.0,'maxdim':768,'neg':False,'variant':'fp16','extra':{},'ram_gb':{'float16':2.6,'float32':5.2}},
    'sdxl_anime':  {'cls':'StableDiffusionXLPipeline','model':'sca255/nova_xl_lcm','steps':8,'guidance':1.5,'maxdim':1024,'neg':True,'variant':'fp16','extra':{},'ram_gb':{'float16':7.4,'float32':14.7}},
    'sdxl_light':  {'cls':'StableDiffusionXLPipeline','model':'stabilityai/sdxl-turbo','steps':4,'guidance':0.0,'maxdim':768,'neg':False,'variant':'fp16','extra':{},'ram_gb':{'float16':6.9,'float32':13.9}},
    'animagine':   {'cls':'StableDiffusionXLPipeline','model':'cagliostrolab/animagine-xl-4.0','steps':25,'guidance':7.0,'maxdim':1024,'neg':True,'variant':None,'extra':{},'ram_gb':{'float16':6.9,'float32':6.9}},
    'sana06':      {'cls':'SanaSprintPipeline','model':'Efficient-Large-Model/Sana_Sprint_0.6B_1024px_diffusers','steps':4,'guidance':4.5,'maxdim':1024,'neg':False,'variant':None,'extra':{'intermediate_timesteps':None,'clean_caption':True},'ram_gb':{'float16':7.7,'float32':7.7}},
}
CPU_PRESET = 'sdxl_anime'
CPU = {'pipe': None, 'key': None, 'loading': False, 'error': None, 'engine': None, 'dtype': None}

def set_cpu_preset(name):
    global CPU_PRESET
    if name in CPU_PRESETS and name != CPU_PRESET:
        CPU_PRESET = name; CPU['pipe'] = None; CPU['key'] = None; CPU['error'] = None

def _free_ram(tag=''):
    import gc
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        import psutil
        vm = psutil.virtual_memory()
        print('[i] RAM%s %.1f 总 / %.1f 空闲' % (tag, vm.total/1024**3, vm.available/1024**3))
    except Exception:
        pass

def _avail_ram_gb():
    try:
        import psutil
        return psutil.virtual_memory().available / 1024**3
    except Exception:
        return 99.0

def _load_cpu_diffusers(pr):
    import torch
    from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline
    try:
        from diffusers import SanaSprintPipeline
    except ImportError:
        SanaSprintPipeline = None
    CLS = {'StableDiffusionPipeline': StableDiffusionPipeline,
           'StableDiffusionXLPipeline': StableDiffusionXLPipeline,
           'SanaSprintPipeline': SanaSprintPipeline}[pr['cls']]
    if CLS is None:
        raise RuntimeError('diffusers 无 %s，换 cpu_preset' % pr['cls'])
    kw = {'use_safetensors': True}
    pr_variant = pr.get('variant')
    # ★ CPU 上 fp32 才是快路径：PyTorch 的 fp16 matmul 在 CPU 上是逐算子升 fp32 模拟的，
    #   实测比 fp32 慢数倍（sd_turbo 4步512px 用 fp16 跑了 20+ 分钟没出图）。
    #   所以顺序是 fp32 优先；只有 fp32 过不了 RAM 闸门才退 fp16（省一半内存、换速度）。
    #   有 CUDA 时反过来：fp16 又快又省。
    try:
        _has_cuda = torch.cuda.is_available()
    except Exception:
        _has_cuda = False
    order = ('float16', 'float32') if _has_cuda else ('float32', 'float16')
    last = None
    for dt in order:
        need = pr.get('ram_gb', {}).get(dt)
        if need and _avail_ram_gb() < need + 1.5:
            last = RuntimeError('%s 权重需 ~%.1fGB RAM，当前只剩 %.1fGB -> 换更小的 cpu_preset（sd_turbo 只要 2.6GB）'
                                % (dt, need, _avail_ram_gb()))
            print('[x]', str(last), '-> 试下一个 dtype')
            continue
        kw2 = dict(kw)
        if pr_variant and dt == 'float16':
            kw2['variant'] = pr_variant
        try:
            try:
                p = CLS.from_pretrained(pr['model'], dtype=getattr(torch, dt), **kw2)
            except TypeError:        # 老版 diffusers 只认 torch_dtype
                p = CLS.from_pretrained(pr['model'], torch_dtype=getattr(torch, dt), **kw)
            print('[+] CPU 权重 dtype=%s' % dt)
            CPU['dtype'] = dt
            return p
        except Exception as e:
            last = e
            msg = str(e)
            if kw2.get('variant') and ('fp16' in msg or 'not found' in msg or 'safetensors' in msg):
                print('[!] variant=fp16 不可用(%s) -> 去掉 variant 重试' % msg[:120])
                pr_variant = None
                continue
            _free_ram(' (重载前)')
    raise last

def _load_cpu_openvino(pr):
    from optimum.intel import OVDiffusionPipeline
    return OVDiffusionPipeline.from_pretrained(pr['model'], compile=True)

def load_cpu_pipe():
    if CPU['pipe'] is not None and CPU['key'] == CPU_PRESET:
        return CPU['pipe']
    if CPU['loading']:
        raise RuntimeError('CPU model loading, retry in 120s')
    CPU['loading'] = True; CPU['error'] = None
    pr = CPU_PRESETS[CPU_PRESET]
    try:
        if CPU['pipe'] is not None:
            del CPU['pipe']; CPU['pipe'] = None; CPU['key'] = None
            _free_ram(' (切档)')
        p = None
        if CPU_ENGINE == 'openvino':
            try:
                p = _load_cpu_openvino(pr); CPU['engine'] = 'openvino'; CPU['dtype'] = 'ov'
            except Exception as e:
                print('[!] OpenVINO 档失败(%s: %s) -> 降级 diffusers fp16'
                      % (type(e).__name__, str(e)[:160]))
                p = None
        if p is None:
            p = _load_cpu_diffusers(pr); CPU['engine'] = 'diffusers-' + (CPU.get('dtype') or 'fp32')
        p.set_progress_bar_config(disable=True)
        # 纯 CPU runtime 没有 accelerator，offload 会抛 ValueError；只有真 CUDA 时才挂 offload
        try:
            import torch as _t
            _accel = _t.cuda.is_available()
        except Exception:
            _accel = False
        if _accel:
            try:
                p.enable_model_cpu_offload(); print('[+] offload: model_cpu (权重常驻 RAM，逐组件上卡)')
            except Exception:
                try:
                    p.enable_sequential_cpu_offload(); print('[+] offload: sequential_cpu')
                except Exception:
                    print('[!] offload 全失败，按纯 CPU 跑')
        else:
            try:
                p.to('cpu')
            except Exception as e:
                print('[!] .to(cpu) 失败: %s' % str(e)[:120])
            # 纯 CPU：attention slicing / vae tiling 省的是显存，CPU 上只会拖慢，默认不开。
            # 真要省内存再手动打开（CPU_SLICING=1）。
            if CPU_SLICING:
                try:
                    p.enable_attention_slicing()
                    p.enable_vae_slicing(); p.enable_vae_tiling()
                    print('[i] CPU_SLICING=1 已开 attention/vae slicing（慢但省内存）')
                except Exception:
                    pass
            else:
                print('[i] 纯 CPU：不开 slicing（省显存无用、纯减速）')
            try:
                import torch as _t
                _n = _t.get_num_threads()
                _avail = os.cpu_count() or 2
                if _n < _avail:
                    _t.set_num_threads(_avail)
                print('[i] torch 线程 %d -> %d (cpu_count=%s)' % (_n, _t.get_num_threads(), _avail))
            except Exception as e:
                print('[!] set_num_threads 失败: %s' % str(e)[:100])
        _free_ram(' (加载后)')
        CPU['pipe'] = p; CPU['key'] = CPU_PRESET
        print('[+] CPU 档就绪 | preset=%s engine=%s %s 默认 %d步 cfg%.1f <=%dpx'
              % (CPU_PRESET, CPU['engine'], pr['model'], pr['steps'], pr['guidance'], pr['maxdim']))
        return p
    except Exception as e:
        CPU['error'] = ('%s: %s' % (type(e).__name__, str(e)))[:300]
        raise
    finally:
        CPU['loading'] = False

def set_job(jid, **kw):
    with LOCK:
        JOBS.setdefault(jid, {'status':'queued','created':time.time()}).update(kw)

def get_job(jid):
    with LOCK:
        j = JOBS.get(jid)
        return dict(j) if j else None

# ---- 视频（懒加载，第一次请求才下 ~10-20GB）----
VID_MODEL = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
VIDEO = {'pipe': None, 'loading': False, 'error': None}

def load_video_pipe():
    if VIDEO['pipe'] is not None:
        return VIDEO['pipe']
    if VIDEO['loading']:
        raise RuntimeError('video model loading, retry in 60s')
    VIDEO['loading'] = True
    try:
        import torch
        from diffusers import WanPipeline, AutoencoderKLWan
        vae = AutoencoderKLWan.from_pretrained(VID_MODEL, subfolder='vae', torch_dtype=torch.float32)
        p = WanPipeline.from_pretrained(VID_MODEL, vae=vae, torch_dtype=torch.float16)
        try:
            p.enable_model_cpu_offload(); p.enable_vae_slicing()
        except Exception:
            p.to('cuda')
        VIDEO['pipe'] = p
        print('[+] video pipeline ready')
        return p
    except Exception as e:
        VIDEO['error'] = '%s: %s' % (type(e).__name__, e)
        raise RuntimeError(VIDEO['error'])
    finally:
        VIDEO['loading'] = False

def _is_oom(e):
    s = str(e).lower()
    return 'out of memory' in s or 'cuda oom' in s

def render_image(d):
    import torch
    p = (d.get('prompt') or '').strip()
    if p and not p.startswith('masterpiece'):
        p = 'masterpiece, best quality, score_7, ' + p
    neg = d.get('negative') or ('worst quality, low quality, score_1, score_2, score_3, '
                                'artist name, blurry, jpeg artifacts, chromatic aberration')
    if HAS_GPU:
        if pipe is None:
            raise RuntimeError('run cell 3 first')
        gd = (globals().get('GPU_DEFAULTS') or {}).get(globals().get('GPU_MODEL')) or {}
        w = int(d.get('width') or gd.get('w') or 832)
        h = int(d.get('height') or gd.get('h') or 1216)
        steps = int(d.get('steps') or gd.get('steps') or 28)
        seed = int(d.get('seed') or 0)
        kw0 = {}
        if gd.get('guidance') is not None:
            kw0['guidance_scale'] = float(gd['guidance'])
        err = None
        for attempt in range(3):
            try:
                out = pipe(prompt=p, negative_prompt=neg, height=h, width=w,
                           num_inference_steps=steps,
                           generator=torch.Generator('cuda').manual_seed(seed), **kw0)
                img = out.images[0]
                m = float(np.asarray(img.convert('RGB')).mean())
                if m < 6:
                    raise RuntimeError('black image (mean=%.1f) - rerun cell 3' % m)
                buf = io.BytesIO(); img.save(buf, format='PNG')
                return buf.getvalue(), 'png'
            except Exception as e:
                err = e
                if _is_oom(e) and (w > 512 or h > 512):
                    import gc; gc.collect(); torch.cuda.empty_cache()
                    w = max(512, (w*4//5)//8*8); h = max(512, (h*4//5)//8*8)
                    print('[!] OOM -> 降到 %dx%d 重试' % (w, h))
                    continue
                raise
        raise err
    else:
        if d.get('cpu_preset'):
            set_cpu_preset(str(d['cpu_preset']))
        pr = CPU_PRESETS[CPU_PRESET]; ov = load_cpu_pipe()
        dim = pr['maxdim']
        w = min(int(d.get('width') or dim), dim); h = min(int(d.get('height') or dim), dim)
        w -= w % 8; h -= h % 8
        kw = dict(prompt=p, height=h, width=w,
                  num_inference_steps=int(d.get('steps') or pr['steps']),
                  guidance_scale=float(d.get('guidance') if d.get('guidance') is not None else pr['guidance']),
                  generator=torch.Generator('cpu').manual_seed(int(d.get('seed') or 0)))
        kw.update(pr.get('extra') or {})
        if pr.get('neg'):
            kw['negative_prompt'] = neg
        _st = {'t0': time.time()}; _jid = d.get('_jid') or ''
        def _prog(pipe_obj, step, timestep, cb_kwargs):
            if _jid and (step == 0 or step % 2 == 0):
                set_job(_jid, progress='%d/%d' % (step + 1, kw['num_inference_steps']),
                        elapsed=round(time.time() - _st['t0'], 1))
        kw['callback_on_step_end'] = _prog
        img = ov(**kw).images[0]
        m = float(np.asarray(img.convert('RGB')).mean())
        if m < 6:
            raise RuntimeError('black image on CPU (mean=%.1f) -> 换 cpu_preset' % m)
        buf = io.BytesIO(); img.save(buf, format='PNG')
        return buf.getvalue(), 'png'

def render_video(d):
    import torch, subprocess
    from diffusers.utils import export_to_video, load_image
    vp = load_video_pipe()
    neg = d.get('negative') or ('模糊不清，字幕，最差质量，低质量，JPEG压缩残留，多余的手指')
    kw = dict(prompt=d.get('prompt') or '', negative_prompt=neg,
              height=int(d.get('height') or 704), width=int(d.get('width') or 1280),
              num_frames=int(d.get('frames') or 121), guidance_scale=float(d.get('guidance') or 5.0),
              num_inference_steps=int(d.get('steps') or 50),
              generator=torch.Generator('cuda').manual_seed(int(d.get('seed') or 0)))
    src = d.get('image_path') or ''
    if src and os.path.exists(src):
        kw['image'] = load_image(src)
    frames = vp(**kw).frames[0]
    raw = os.path.join(OUT_DIR, 'v_%d_raw.mp4' % int(time.time()))
    export_to_video(frames, raw, fps=int(d.get('fps') or 24))
    final = raw.replace('_raw.mp4', '.mp4')
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',raw,'-c:v','libx264','-crf','23',
                    '-preset','fast','-movflags','+faststart',final], check=True)
    try: os.remove(raw)
    except OSError: pass
    return open(final,'rb').read(), 'mp4'

def worker():
    while True:
        jid, kind, d = TASK_Q.get()
        set_job(jid, status='running', kind=kind)
        t0 = time.time()
        try:
            blob, ext = (render_video(d) if kind == 'video' else render_image(d))
            path = os.path.join(OUT_DIR, '%s.%s' % (jid, ext))
            open(path, 'wb').write(blob)
            set_job(jid, status='done', seconds=round(time.time()-t0,1),
                    path=path, bytes=len(blob), ext=ext)
        except Exception as e:
            set_job(jid, status='error', error=('%s: %s' % (type(e).__name__, e))[:300])
        finally:
            TASK_Q.task_done()

def submit(kind, d):
    jid = uuid.uuid4().hex
    d = dict(d); d['_jid'] = jid          # 渲染分支回写进度要用
    set_job(jid, status='queued', kind=kind)
    TASK_Q.put((jid, kind, d))
    return jid

@app.route('/health')
def health():
    return jsonify(ok=True, gpu=HAS_GPU, gpu_name=GPU_NAME, vram_gb=round(VRAM_GB,1),
                   pending=TASK_Q.qsize(), jobs=len(JOBS), out=OUT_DIR, gpu_model=globals().get('GPU_MODEL'),
                   video_loaded=VIDEO['pipe'] is not None, video_error=VIDEO['error'],
                   cpu_ready=CPU['pipe'] is not None, cpu_error=CPU['error'],
                   cpu_engine=CPU['engine'], cpu_engine_mode=CPU_ENGINE, cpu_slicing=CPU_SLICING,
                   cpu_preset=CPU_PRESET, cpu_presets=list(CPU_PRESETS))

@app.route('/cpu_presets')
def cpu_presets_ep():
    return jsonify(current=CPU_PRESET, presets={k: v['model'] for k, v in CPU_PRESETS.items()})

@app.route('/generate', methods=['POST'])
def generate():
    d = request.get_json(force=True)
    if not (d.get('prompt') or '').strip():
        return jsonify(error='prompt required'), 400
    kind = d.get('kind') or 'image'
    if kind not in ('image','video'):
        return jsonify(error='kind must be image|video'), 400
    if kind == 'video' and not HAS_GPU:
        return jsonify(error='video needs GPU'), 400
    return jsonify(id=submit(kind, d), queued=TASK_Q.qsize())

@app.route('/batch', methods=['POST'])
def batch():
    d = request.get_json(force=True)
    prompts = d.get('prompts') or []
    if not prompts:
        return jsonify(error='prompts required'), 400
    n = int(d.get('n_per_prompt') or 1)
    kind = d.get('kind') or 'image'
    ids = []
    for pr in prompts:
        for k in range(n):
            dd = dict(d); dd['prompt'] = pr
            if d.get('seed'):
                dd['seed'] = int(d['seed']) + k
            ids.append(submit(kind, dd))
    return jsonify(ids=ids, count=len(ids), queued=TASK_Q.qsize())

@app.route('/status/<jid>')
def status(jid):
    j = get_job(jid)
    if not j:
        return jsonify(error='unknown id'), 404
    return jsonify(j)

@app.route('/image/<jid>')
@app.route('/result/<jid>')
def result(jid):
    j = get_job(jid)
    if not j or j.get('status') != 'done':
        return jsonify(error='not ready'), 404
    mime = 'video/mp4' if j.get('ext') == 'mp4' else 'image/png'
    return send_file(j['path'], mimetype=mime)

@app.route('/upload', methods=['POST'])
def upload():
    f = request.files.get('file')
    if not f:
        return jsonify(error='file field required'), 400
    name = re.sub(r'[^A-Za-z0-9_.-]', '_', f.filename or 'ref.png')
    path = os.path.join(OUT_DIR, 'up_' + name)
    f.save(path)
    return jsonify(ok=True, path=path)

@app.route('/flush')
def flush():
    flush_to_drive()
    return jsonify(ok=True, drive=DRIVE_OK)

# 幂等启动：重跑本格不再报 Address already in use（旧 Flask 活着就复用）
try:
    urllib.request.urlopen('http://127.0.0.1:7860/health', timeout=3)
    print('[i] Flask 已在 7860 运行，复用旧实例（模型走全局 pipe，仍然生效）')
except Exception:
    threading.Thread(target=worker, daemon=True).start()
    threading.Thread(target=lambda: app.run(host='0.0.0.0', port=7860, debug=False, use_reloader=False),
                     daemon=True).start()

# 半截下载的二进制会 EACCES：用 --version 实测可用性，坏就重下；chmod 无条件执行
BIN = '/usr/local/bin/cloudflared'
def _cf_ok():
    try:
        return subprocess.run([BIN,'--version'], capture_output=True, timeout=20).returncode == 0
    except Exception:
        return False
if not _cf_ok():
    subprocess.run(['rm','-f',BIN])
    subprocess.run(['wget','-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O',BIN], check=True)
os.chmod(BIN, 0o755)
subprocess.run(['pkill','-f','cloudflared tunnel'], capture_output=True); time.sleep(1)
proc = subprocess.Popen([BIN,'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            URL = m.group(0); break

print('=' * 64)
print('PUBLIC URL:', URL)
print('=' * 64)
print("存到本机:  echo '%s' > ~/.dsh/.colab-url" % URL)
print('本机 CLI:  scripts/colab_gen.sh "prompt"')

In [ ]:
#@title 5. 自测一张（640x960@22，白衬衫同款配置）
import requests, time, io
from PIL import Image
r = requests.post(URL + '/generate', json={
    'prompt': '1girl, solo, silver long hair, blue eyes, white shirt, soft window light, detailed skin',
    'width': 832, 'height': 1216, 'steps': 28, 'seed': 4242})
jid = r.json()['id']; print('job', jid)
t0 = time.time()
s = {}
while time.time() - t0 < 600:
    s = requests.get(URL + '/status/' + jid, timeout=30).json()
    st = s.get('status')
    if st in ('done', 'error'):
        print(st, s.get('seconds',''), (s.get('error') or '')[:150]); break
    time.sleep(5)
if s.get('status') == 'done':
    img = Image.open(io.BytesIO(requests.get(URL + '/result/' + jid, timeout=120).content))
    print('出图:', img.size, '%.0fs' % s.get('seconds', 0), '| model=%s' % globals().get('GPU_MODEL'))
    display(img)

In [ ]:
#@title 6. 批量出图（改 PROMPTS 一次提交，服务端排队串行出）
PROMPTS = [
    '1girl, solo, silver hair, standing in rain, city night, neon reflections',
    '1girl, solo, red kimono, cherry blossoms, sunlight, bokeh',
    '1girl, solo, black hair, reading in library, warm lamp light',
]
N_PER_PROMPT = 2
W, H, STEPS, SEED = 832, 1216, 28, 1000

import requests, time
r = requests.post(URL + '/batch', json={'prompts': PROMPTS, 'n_per_prompt': N_PER_PROMPT,
                                        'width': W, 'height': H, 'steps': STEPS, 'seed': SEED})
ids = r.json()['ids']
per = 30 if HAS_GPU else 300
print('[+] 入队 %d 张，预计 ~%.0f 分钟' % (len(ids), len(ids)*per/60))
t0 = time.time()
while time.time() - t0 < len(ids)*per*4 + 600:
    st = [get_job(i) or {} for i in ids]
    d = sum(1 for s in st if s.get('status') == 'done')
    e = sum(1 for s in st if s.get('status') == 'error')
    print('  %d+%de/%d  %.1fmin' % (d, e, len(ids), (time.time()-t0)/60))
    if d + e >= len(ids):
        break
    time.sleep(10)
flush_to_drive()
print('[!] /content 会话结束清空，产出已搬 Drive')

In [ ]:
#@title 7. 视频（可选）：Wan 2.2 TI2V-5B，首次触发下 ~10-20GB 权重
MAKE_VIDEO = False
SRC_IMAGE = ''      # Colab 端静帧路径（cell 6 产出的 /content/out/xxx.png）；留空=文生视频
VID_PROMPT = 'gentle camera push in, hair swaying, soft light shift'
VID_FRAMES = 121    # 121帧@24fps=5秒；49 帧省一半时间
VID_STEPS = 50

if not MAKE_VIDEO:
    print('[i] MAKE_VIDEO=False，跳过')
elif not HAS_GPU:
    print('[!] 视频需要 GPU')
else:
    import requests, time, os, gc, torch
    # 视频要整份显存：先释放图像模型（之后要出图，重跑 cell 3 即可）
    if 'pipe' in globals() and pipe is not None:
        globals()['pipe'] = None
        gc.collect(); torch.cuda.empty_cache()
        print('[i] 已释放图像模型腾显存给视频')
    payload = {'kind':'video','prompt':VID_PROMPT,'frames':VID_FRAMES,'steps':VID_STEPS,
               'height':704,'width':1280,'fps':24,'seed':42}
    if SRC_IMAGE and os.path.exists(SRC_IMAGE):
        payload['image_path'] = SRC_IMAGE
    r = requests.post(URL + '/generate', json=payload, timeout=60)
    jid = r.json()['id']; print('job', jid, '（首次先下权重）')
    t0 = time.time(); s = {}
    while time.time() - t0 < 2400:
        s = requests.get(URL + '/status/' + jid, timeout=30).json()
        if s.get('status') in ('done','error'):
            print(s.get('status'), s.get('seconds',''), (s.get('error') or '')[:150]); break
        time.sleep(15)
    if s.get('status') == 'done':
        mp4 = requests.get(URL + '/result/' + jid, timeout=300).content
        out = os.path.join(OUT_DIR, 'vid_%s.mp4' % jid[:8])
        open(out,'wb').write(mp4)
        print('[+]', out, '%.1fMB' % (len(mp4)/1024**2))
        flush_to_drive()

## 跑法与排错

**正常流程**：运行时改 GPU(T4) → 依次跑 1→2→3→4 → 出 URL 存本机 → 第 5 格自测 → 第 6 格批量。

**CPU 档（无 GPU 时）**：cell 4 自动分流，默认 `CPU_PRESET='sdxl_anime'`（nova_xl_lcm fp16，7.3GB 仓库带 fp16 变体）。可选 `sd_turbo`(最快) / `sdxl_light` / `sana06` / `animagine`(25步最慢但最好)。客户端传 `cpu_preset` 即切档，引擎状态看 `/health` 的 `cpu_engine`。

**cell 3（GPU 模型）**：默认 `GPU_MODEL='animagine'`（SDXL 动漫 fp16，实测 ~30s/张@832x1216）。
- 要 Anima 质感 → 改 `'anima'`（fp32 慢速，512x768 稳）；要更快 → `'nova_lcm'`（8步 ~10s，画质略降）
- 加载后自动出 512 测试图，mean<6 判黑图并提示换档

**常见问题**：
| 现象 | 处理 |
|---|---|
| `tunnel unreachable` / Cloudflare 1033 | runtime 重启过，重跑 cell 4，更新 ~/.dsh/.colab-url |
| 出图 `black image` | SDXL fp16 不该黑；发生则改 GPU_MODEL 重跑 cell 3 |
| OOM | 已自动降档重试；还 OOM 就重跑 cell 3（清碎片）或降到 640×960 |
| RAM 爆 | 别反复重载模型；一个会话只跑一次 cell 3 |
| CPU 档 `Could not import module 'OVDiffusionPipeline'` | **旧版 notebook 才有**；本版 CPU 档已改纯 diffusers fp16，不再走 optimum-intel。出现即说明跑的是旧代码 → File→Open notebook→GitHub 开新副本 |
| 排队慢 | 单 worker 是故意的（并发必 OOM），批量提交排队消化 |
| cell 4 报 `Address already in use`/`PermissionError cloudflared` | 直接重跑 cell 4 即可（已幂等：复用旧 Flask、坏二进制自动重下） |
| 改了 GitHub 但跑的还是旧代码 | **F5 不会重拉 GitHub**。用 File→Open notebook→GitHub→贴 URL（main 分支）开新副本 |

**会话现实**：免费档 3.5-6h、/content 清空、12.6GB RAM。产出走 Drive 或本机下载。